# Chapter 17 -- Exercises: Loops, Goals, and Iterations

Three exercises, each framed around a course concept rather than a finance formula: the **hook** (a checkable success gate), the **loop** (sign-guided iteration), and a **stress engine** with a decision hook. For each, fill in the function body (replace `raise NotImplementedError`) and then run the test cell -- the `assert`s are your success gate. Difficulty tags: `[B]` beginner, `[I]` intermediate, `[A]` advanced.

Use only `numpy` / `pandas` if you wish; none is strictly required. All credit figures are illustrative and hypothetical.

In [1]:
import numpy as np
import pandas as pd

## Exercise [B] -- The success-gate HOOK

A loop is only as good as the hook that decides when it is done. Implement `within_tolerance(value, tol)` returning `True` iff the magnitude of `value` is *strictly* below `tol`. This single predicate is what stops every loop in this chapter.

In [2]:
def within_tolerance(value, tol):
    """Return True iff |value| < tol. The success-gate HOOK."""
    return abs(value) < tol


In [3]:
assert within_tolerance(0.0008, 1e-3) is True
assert within_tolerance(0.008, 1e-3) is False
assert within_tolerance(-0.0005, 1e-3) is True
assert within_tolerance(0.001, 1e-3) is False   # strict: not below tol
print("Exercise B passed: the hook gates correctly.")

Exercise B passed: the hook gates correctly.


## Exercise [I] -- The sign-guided root-finding LOOP

Implement `find_root(f, lo, hi, tol, max_iters)`: a **derivative-free** loop that returns an `x` with `|f(x)| < tol`, given a bracket `[lo, hi]` where `f(lo)` and `f(hi)` have opposite signs. Use only evaluations of `f` and the **sign** of the residual (bisection or regula falsi) -- no derivatives, no `f'`. Then watch the *same* loop solve a finance instance (the IRR) just by passing it `NPV` instead of `f`.

In [4]:
def find_root(f, lo, hi, tol=1e-3, max_iters=100):
    """Sign-guided, derivative-free bisection root finder."""
    flo, fhi = f(lo), f(hi)
    if flo * fhi > 0:
        raise ValueError("f(lo) and f(hi) must have opposite signs")
    for _ in range(max_iters):
        mid = 0.5 * (lo + hi); fm = f(mid)
        if abs(fm) < tol:
            return mid
        if flo * fm < 0:
            hi, fhi = mid, fm
        else:
            lo, flo = mid, fm
    return 0.5 * (lo + hi)


In [5]:
def f(x):
    return x**3 - x - 2

root = find_root(f, 1.0, 2.0, tol=1e-3, max_iters=100)
assert abs(f(root)) < 1e-3, f"root residual too large: {f(root)}"
assert abs(root - 1.5213797) < 1e-2, f"root off target: {root}"

# SAME loop, finance instance: x -> r, f -> NPV. IRR of [-100, 60, 60].
def npv(r):
    return -100 + 60 / (1 + r) + 60 / (1 + r) ** 2

irr = find_root(npv, 0.0, 1.0, tol=0.05, max_iters=100)
assert abs(npv(irr)) < 0.05, f"NPV residual too large: {npv(irr)}"
assert abs(irr - 0.1307) < 0.02, f"IRR off target: {irr}"
print(f"Exercise I passed: root ~ {root:.5f}, IRR ~ {irr:.4f} (same loop).")

Exercise I passed: root ~ 1.52148, IRR ~ 0.1309 (same loop).


## Exercise [A] -- The credit-risk stress engine + decision hook

Implement `stress_expected_loss(base, stressed, ead)` using `EL = PD * LGD * EAD`, where `base` and `stressed` are `(PD, LGD)` tuples. Return a dict with keys `'base_el'`, `'stressed_el'` (dollars) and `'base_bps'`, `'stressed_bps'` (basis points of EAD). Also implement `decision_gate(base_bps, stressed_bps, ...)` returning `'lend'` / `'modify'` / `'decline'`. The tests reconcile against the chapter's illustrative Meridian figures (base $0.40m / 100 bps, stressed $1.98m / 495 bps) and check the decision is `'modify'`.

In [6]:
def stress_expected_loss(base, stressed, ead):
    """EL = PD * LGD * EAD for base and stressed (PD, LGD)."""
    bpd, blgd = base; spd, slgd = stressed
    base_el = bpd * blgd * ead; stressed_el = spd * slgd * ead
    return {"base_el": base_el, "stressed_el": stressed_el,
            "base_bps": base_el / ead * 1e4, "stressed_bps": stressed_el / ead * 1e4}

def decision_gate(base_bps, stressed_bps, lend_max=150, decline_min=500):
    if stressed_bps >= decline_min:
        return "decline"
    if base_bps <= lend_max and stressed_bps <= lend_max:
        return "lend"
    return "modify"


In [7]:
ead = 40e6
res = stress_expected_loss(base=(0.025, 0.40), stressed=(0.09, 0.55), ead=ead)
assert abs(res['base_el'] - 0.40e6) < 1.0, res['base_el']
assert abs(res['stressed_el'] - 1.98e6) < 1.0, res['stressed_el']
assert round(res['base_bps']) == 100, res['base_bps']
assert round(res['stressed_bps']) == 495, res['stressed_bps']
assert decision_gate(res['base_bps'], res['stressed_bps']) == 'modify'
assert decision_gate(20, 90) == 'lend'        # both comfortably low
assert decision_gate(120, 650) == 'decline'   # severe stressed loss
print("Exercise A passed: EL reconciles; gate returns modify / lend / decline.")

Exercise A passed: EL reconciles; gate returns modify / lend / decline.


## Done

You implemented the three pieces every loop in this chapter rests on: a **hook** ([B]) that decides *done*, a **loop** ([I]) that moves a state variable using feedback alone (and works unchanged on a finance problem), and a **stress engine + decision hook** ([A]) that turns arithmetic into a defensible verdict. The loop is the unit of design; the hook is where trust lives. All credit figures are illustrative and hypothetical.